# Project test notebook

This notebook is the single test-suite guide for the recommender repository. It explains the test layers, shows how to run focused checks, and offers one final cell to run all tests from the active kernel.

**Environment:** use the project Python 3.11 + Java 21 Linux/WSL environment for Spark tests. The recorded Windows Python 3.14 run had Spark JVM startup errors. Tests use small synthetic fixtures; passing tests prove code behavior on those fixtures, not model quality, large-scale performance, AWS deployment, or business impact.

Run from the repository checkout. The full suite can take a few minutes and starts local Spark sessions.

## What the test suite covers

| Test file | Behavior under test |
|---|---|
| `test_prepare.py` | Required interaction filtering, duplicate/sparse handling, and chronological train/validation/test split. |
| `test_prepare_tower_data.py` | Deterministic training-only ID mappings and exclusion of unseen held-out IDs. |
| `test_sampling.py` | Repeatable negative sampling, exclusion of training positives, dense-history behavior, and invalid indices. |
| `test_two_tower.py` | Embedding shapes, finite dot-product scores, and BPR gradient direction. |
| `test_train_two_tower.py` | Tiny synthetic fit, checkpoint reload, and selection of the lowest validation-loss epoch. |
| `test_evaluate_two_tower.py` | Hand-computed Recall/Precision/NDCG, seen-item removal, split isolation, and cohort reporting. |
| `test_popularity.py` | Popularity ordering, stable ties, and per-user hit-rate behavior. |
| `test_collaborative_filtering.py` | ALS factor scoring, source-ID alignment, archive behavior, and comparison aggregation. |
| `test_xgboost_ranker.py` | Candidate sampling, training-only feature construction, and a small grouped ranker fit. |
| `test_model_registration.py` | Validation-only quality gate; rejects regression, insufficient NDCG lift, and non-finite results. |
| `test_api.py` | Ready/model-info/recommendation behavior, seen-item filtering, unknown-user policy, bundle checksums, request limits, and Redis outage fallback. |
| `test_event_schema.py` | Valid click/view JSON events and rejection of malformed schemas. |
| `test_experiment_tracking.py` | Local MLflow parameter, metric, and artifact persistence. |

The test code is in `tests/`; implementation code is under `src/`. This table summarizes intent; inspect the assertions in the test files for exact behavior.

In [1]:
from pathlib import Path
import os
import sys
import platform

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").is_file():
    raise RuntimeError("Open this notebook from within the project checkout.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")

# Choose one: "full" (default), "fast", or "spark".
RUN_MODE = "full"
print("Project:", ROOT)
print("Python:", platform.python_version(), platform.platform())
print("Kernel:", sys.executable)
print("Run mode:", RUN_MODE)
print("Spark tests are best run with Python 3.11 and Java 21 under Linux/WSL.")

Project: C:\Users\aksha\PycharmProjects\EndtoEndMLProject
Python: 3.14.7 Windows-11-10.0.26300-SP0
Kernel: C:\Users\aksha\PycharmProjects\EndtoEndMLProject\venv\Scripts\python.exe
Run mode: full
Spark tests are best run with Python 3.11 and Java 21 under Linux/WSL.


## Run focused, fast tests

These tests avoid Spark startup and are useful while changing model math, API behavior, or the validation gate. Restart the kernel if a prior Spark test left the JVM in a broken state.

In [ ]:
import pytest

FAST_TESTS = [
    "tests/test_two_tower.py",
    "tests/test_sampling.py",
    "tests/test_api.py",
    "tests/test_model_registration.py",
]
if RUN_MODE == "fast":
    fast_exit = pytest.main(["-q", *FAST_TESTS])
    print("Focused test exit code:", fast_exit)
else:
    print("Skipped; set RUN_MODE = 'fast' to run this quick subset.")

## Run Spark/data integration tests

This group checks real local Spark DataFrame behavior on small fixtures. It does not need the Amazon dataset. It does need a compatible Java/Spark runtime. Spark startup errors on Windows/Python 3.14 should be treated as an environment issue first; the project has a successful WSL Python 3.11 + Java 21 test record.

In [ ]:
SPARK_TESTS = [
    "tests/test_prepare.py",
    "tests/test_prepare_tower_data.py",
    "tests/test_popularity.py",
    "tests/test_evaluate_two_tower.py",
    "tests/test_collaborative_filtering.py",
]
if RUN_MODE == "spark":
    spark_exit = pytest.main(["-q", *SPARK_TESTS])
    print("Spark test exit code:", spark_exit)
else:
    print("Skipped; set RUN_MODE = 'spark' to run only the Spark tests.")

## Run the complete suite

This invokes pytest inside the notebook's active Python process (no child shell or separate interpreter). It covers all test files, including event schema, tracking, ranker, and model training. A non-zero pytest exit code means at least one test failed or the environment could not run a test. The captured summary is shown below the cell.

In [ ]:
if RUN_MODE == "full":
    all_exit = pytest.main(["-q", "tests"])
    print("Full suite exit code:", all_exit)
    if int(all_exit) != 0:
        raise AssertionError(f"Pytest suite did not pass: exit code {all_exit}")
else:
    print("Skipped; the selected mode ran in the focused or Spark-only cell.")

## How to read the result

A passing suite checks implementation contracts on controlled examples. It does not establish that a model beats popularity: use the end-to-end notebook's fixed validation comparison for that. It does not establish production readiness: container/cloud checks, access controls, load behavior, rollback, and cost require separate evidence.

For a reproducible record, note the commit, Python/Java/PySpark versions, exact command/targets, test count, elapsed time, and any skipped/failed tests. The historical report in `docs/two_tower_test_report.md` records earlier runs; rerun this notebook to establish the result for the current checkout.

## Browse a test implementation inline

Choose a test module to display its complete source in the notebook. This lets you study the assertions next to the test-layer explanations above without opening the Python file separately.

In [ ]:
from IPython.display import display, Code

TEST_MODULES = sorted(path.name for path in (ROOT / "tests").glob("test_*.py"))
print("Available test modules:", ", ".join(TEST_MODULES))
TEST_TO_INSPECT = "test_api.py"
if TEST_TO_INSPECT not in TEST_MODULES:
    raise KeyError(f"Choose one of: {', '.join(TEST_MODULES)}")
display(Code((ROOT / "tests" / TEST_TO_INSPECT).read_text(encoding="utf-8"), language="python"))